# m23 lab · Voltage mapping

*Open EP Mapping Lab. Education only: not for diagnosis, treatment or any clinical decision.*

A voltage map colors each site by the peak-to-peak amplitude of its electrogram. You compute how unipolar and bipolar amplitudes depend on electrode height and spacing, how a bipole's amplitude depends on the wavefront direction, what noise sets as the lowest readable voltage, how cutoffs trade sensitivity for specificity, and how direction and contact mislabel healthy tissue on a synthetic map.

| Part | What you do | Equation |
| --- | --- | --- |
| A | Amplitude, height and spacing | eq-m23-amplitude |
| B | Wavefront direction | eq-m23-direction |
| C | The noise floor | eq-m23-noise |
| D | Cutoffs | eq-m23-threshold |
| E | A synthetic voltage map | all |

Lengths in mm. The front model follows m03; amplitudes are relative to its source strength unless stated. Tissue voltages, spreads, noise and heights are illustrative; the 0.5-mV and 1.5-mV cutoffs are from Marchlinski and colleagues. All data are synthetic. Each part ends with **checks**.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

try:
    import epml
except ImportError:
    if "google.colab" in sys.modules:
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/raghavendrasri/open-ep-mapping-lab",
                        "open-ep-mapping-lab"], check=True)
        sys.path.insert(0, "open-ep-mapping-lab/labs/python")
    else:
        sys.path.insert(0, str(Path.cwd() / "labs" / "python"))
    import epml

plt.rcParams.update({"figure.figsize": (8, 3.2), "axes.grid": True, "axes.spines.top": False, "axes.spines.right": False})

from epml import voltage as VM

## Part A · Amplitude, height and spacing

In the m03 model the unipolar peak-to-peak is $A/a$ with $a = w + h$, and a bipole's tends to
$\tfrac98 A d\lvert\cos\theta\rvert/a^2$ when its spacing is small next to $a$, and to the unipolar value when it is large
(eq-m23-amplitude). So a raised electrode loses bipolar amplitude faster than unipolar.

In [ ]:
print(" h (mm)  unipolar  bipolar d=0.2  bipolar d=2  (small-spacing formula, d=2)")
for h in (0.5, 1.0, 2.0, 3.0):
    a = 0.3 + h
    print(f"{h:6.1f} {VM.unipolar_pp(a):9.3f} {VM.bipolar_pp(a, 0.2):12.4f} {VM.bipolar_pp(a, 2.0):12.3f} {VM.bipolar_pp_small(a, 2.0):14.3f}")
for name, d in (("bipolar, d = 0.2 mm", 0.2), ("bipolar, d = 2 mm", 2.0)):
    print(f"lifting from 1 to 2 mm keeps {VM.bipolar_pp(2.3, d) / VM.bipolar_pp(1.3, d):.2f} of the {name} amplitude")
print(f"lifting from 1 to 2 mm keeps {VM.unipolar_pp(2.3) / VM.unipolar_pp(1.3):.2f} of the unipolar amplitude")

In [ ]:
# Checks
assert VM.bipolar_pp(2.3, 0.2) / VM.bipolar_pp(1.3, 0.2) < VM.unipolar_pp(2.3) / VM.unipolar_pp(1.3)

## Part B · Wavefront direction

A bipole's amplitude falls as $\lvert\cos\theta\rvert$ in the small-spacing limit (m03). If directions are uniform, the
fraction of readings below a fraction $q$ of the aligned amplitude is $\tfrac2\pi\arcsin q$ (eq-m23-direction). With
$q = 1/3$, the ratio of the 0.5-mV and 1.5-mV cutoffs, healthy tissue at the normal cutoff reads as dense scar for about
one direction in five. The larger of two orthogonal bipoles never drops below $1/\sqrt2$; the largest projection of
the pair barely depends on direction at small spacing, but not at 2 mm.

In [ ]:
rng = np.random.default_rng(23)
angles = rng.uniform(0, np.pi, 300)
a = 1.3
for d in (0.2, 2.0):
    ref = VM.bipolar_pp(a, d, 0.0)
    single = np.array([VM.bipolar_pp(a, d, t) for t in angles]) / ref
    pair = np.array([max(VM.bipolar_pp(a, d, t), VM.bipolar_pp(a, d, np.pi / 2 - t)) for t in angles]) / ref
    field = np.array([VM.field_pp(a, d, t, n_dir=90) for t in angles[:60]]) / ref
    print(f"d = {d} mm: below 1/3 -> one bipole {np.mean(single < 1 / 3):.3f}, pair {np.mean(pair < 1 / 3):.3f}, "
          f"projection {np.mean(field < 1 / 3):.3f}; projection range {field.min():.2f}-{field.max():.2f}")
print("small-spacing formula:", round(VM.below_fraction(1 / 3), 3))

In [ ]:
# Checks
assert abs(VM.below_fraction(1 / 3) - 0.216) < 0.001

## Part C · The noise floor

Even with no signal, the peak-to-peak of noise over a window is not zero. For $n$ independent Gaussian samples of SD
$\sigma$, its expected value is $2\sigma\,\mathrm E[\max]$ (eq-m23-noise), about $5\sigma$ for 100 samples. A voltage map
cannot show amplitudes below that floor. The noise SD here is illustrative.

In [ ]:
sigma = 0.02                                            # mV, illustrative
for n in (10, 50, 100, 500):
    sim = np.mean(np.ptp(rng.normal(0, sigma, (20000, n)), axis=1))
    print(f"n = {n:4d}: expected {VM.noise_pp_expected(n, sigma):.3f} mV, simulated {sim:.3f} mV")

In [ ]:
# Checks
assert 4.9 < VM.noise_pp_expected(100) < 5.1

## Part D · Cutoffs

Model scar and healthy voltages as log-normal (illustrative medians 0.2 and 2.5 mV, equal spread). A cutoff $t$ has
sensitivity $\Phi((\ln t - \ln m_s)/\sigma)$ and specificity $1 - \Phi((\ln t - \ln m_n)/\sigma)$ (eq-m23-threshold); with
equal spreads their sum peaks at $\sqrt{m_s m_n}$. Then lift the electrode by 1 mm and see healthy tissue slide toward
the cutoff.

In [ ]:
ms, mn, sd = 0.2, 2.5, 0.7
for cut in (0.5, VM.youden_cutoff_equal_sd(ms, mn), 1.5):
    sens, spec = VM.threshold_rates(cut, ms, mn, sd, sd)
    print(f"cutoff {cut:.2f} mV: sensitivity {sens:.3f}, specificity {spec:.3f}")
f = VM.bipolar_pp(2.3, 2.0) / VM.bipolar_pp(1.3, 2.0)      # a 2-mm bipole lifted from 1 to 2 mm (part A)
sens, spec = VM.threshold_rates(1.5, ms * f, mn * f, sd, sd)
print(f"after a 1-mm lift (readings x {f:.2f}): at 1.5 mV sensitivity {sens:.3f}, specificity {spec:.3f}")
cuts = np.exp(np.linspace(np.log(0.05), np.log(10), 400))
roc = np.array([VM.threshold_rates(c, ms, mn, sd, sd) for c in cuts])
plt.plot(1 - roc[:, 1], roc[:, 0]); plt.xlabel("1 - specificity"); plt.ylabel("sensitivity"); plt.title("ROC of a voltage cutoff"); plt.show()

In [ ]:
# Checks
assert VM.threshold_rates(1.5, ms * f, mn * f, sd, sd)[1] < VM.threshold_rates(1.5, ms, mn, sd, sd)[1]

## Part E · A synthetic voltage map

A 40 × 40 mm sheet of tissue holds a 10-mm-radius scar whose signals are 10 times smaller. A catheter visits 400 random
sites with random direction and an illustrative height between 0.5 and 2.5 mm; healthy tissue reads 2.5 mV with an
aligned bipole at 1 mm height. Count the healthy sites labeled dense scar (below 0.5 mV) with one bipole and with the
largest projection of a pair.

In [ ]:
n = 400
pos = rng.uniform(-20, 20, (n, 2))
scar = np.linalg.norm(pos - np.array([5.0, 0.0]), axis=1) < 10
theta = rng.uniform(0, np.pi, n)
h = rng.uniform(0.5, 2.5, n)
d = 2.0
ref = VM.bipolar_pp(1.3, d, 0.0)
scale = np.where(scar, 0.1, 1.0) * 2.5 / ref           # mV per model unit
single = np.array([VM.bipolar_pp(0.3 + hh, d, t) for hh, t in zip(h, theta)]) * scale
proj = np.array([VM.field_pp(0.3 + hh, d, t, n_dir=60) for hh, t in zip(h, theta)]) * scale
for name, v in (("one bipole", single), ("largest projection", proj)):
    print(f"{name:18s}: healthy labeled < 0.5 mV {np.mean(v[~scar] < 0.5):.3f}; scar labeled < 0.5 mV {np.mean(v[scar] < 0.5):.3f}")
fig, ax = plt.subplots(1, 2, figsize=(9, 4))
for k, (name, v) in enumerate((("one bipole", single), ("largest projection", proj))):
    sc = ax[k].scatter(pos[:, 0], pos[:, 1], c=np.log10(v), cmap="viridis", s=14, vmin=-1.5, vmax=0.8)
    ax[k].add_patch(plt.Circle((5, 0), 10, fill=False, ls="--")); ax[k].set_title(name); ax[k].set_aspect("equal")
fig.colorbar(sc, ax=ax, label="log10 bipolar voltage (mV)"); plt.show()

In [ ]:
# Checks
assert np.mean(proj[~scar] < 0.5) < np.mean(single[~scar] < 0.5)

## Where to go next

- Add the noise floor from part C to every reading and see how it hides the deepest scar.
- Replace the uniform height with a contact model in which height falls with force (m13).
- m24 interpolates these point values into a map; m26 asks how many points a map needs.